## 0. 환경설정

### Transformer 기반 Ko→En Seq2Seq 번역 (PyTorch)

- 데이터셋: [`shihyunlim/aihub-ko-en-everyday-expression`](https://huggingface.co/datasets/shihyunlim/aihub-ko-en-everyday-expression) (train split만 존재, 약 169만 쌍)
- 모델: `torch.nn.Transformer`(PyTorch 내장 레이어)로 직접 구성한 Encoder-Decoder Transformer
  - 개선: 임베딩 초기화(std=d_model^-0.5), 디코더 임베딩-출력층 weight tying
- 토크나이저: 데이터로 직접 학습하는 Byte-level BPE (ko/en 각각)
- 학습: 1 epoch만 수행 (과제 규칙), AMP(mixed precision)
  - **비선호 학습(unlikelihood training)** 추가: 이미 등장한 토큰의 재생성 확률을 낮춰 반복 루프 억제
- 평가: Loss/Perplexity + BLEU + 정성적 예시 번역
  - **추론 하이퍼파라미터 탐색**: greedy / repetition penalty / no-repeat n-gram / beam search(+length penalty) 설정별 BLEU 비교
- **하이퍼파라미터 탐색**: 실제 런타임(GPU/CPU)에서 후보 설정별 속도를 측정해 표로 보여주고, 사용자가 직접 데이터 규모/모델 크기를 선택하는 셀이 포함되어 있습니다. (Colab에서 GPU 런타임으로 실행하는 것을 권장)


In [16]:
# Colab이면 아래 설치가 필요합니다 (로컬 venv에 이미 있다면 스킵됨)
%pip install -q datasets tokenizers sacrebleu


In [17]:
import math
import random
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore", message=".*nested tensor.*")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


device: cuda
GPU: Tesla T4


## 1. 데이터 로드

In [18]:
from datasets import load_dataset

raw = load_dataset("shihyunlim/aihub-ko-en-everyday-expression")["train"]
print(raw)
print(raw[0])

raw = raw.shuffle(seed=SEED)


Dataset({
    features: ['ko', 'en'],
    num_rows: 1690052
})
{'ko': "'Bible Coloring'은 성경의 아름다운 이야기를 체험 할 수 있는 컬러링 앱입니다.", 'en': "Bible Coloring' is a coloring application that allows you to experience beautiful stories in the Bible."}


## 2. 하이퍼파라미터 탐색

In [19]:
# 2-1. 후보 하이퍼파라미터 설정 정의 + 현재 런타임에서 속도 벤치마크

CANDIDATE_CONFIGS = {
    "tiny":  dict(d_model=128, nhead=4, num_layers=2, dim_feedforward=512,  dropout=0.1, batch_size=128, lr=5e-4),
    "small": dict(d_model=256, nhead=4, num_layers=3, dim_feedforward=1024, dropout=0.1, batch_size=128, lr=4e-4),
    "base":  dict(d_model=256, nhead=8, num_layers=4, dim_feedforward=1024, dropout=0.1, batch_size=64,  lr=3e-4),
    "large": dict(d_model=512, nhead=8, num_layers=6, dim_feedforward=2048, dropout=0.2, batch_size=64,  lr=2e-4),
}

BENCH_VOCAB = 8000
BENCH_SEQ_LEN = 32
BENCH_STEPS = 12

PAD_ID = 0

def build_model(cfg, src_vocab, tgt_vocab):
    return Seq2SeqTransformer(
        src_vocab=src_vocab, tgt_vocab=tgt_vocab,
        d_model=cfg["d_model"], nhead=cfg["nhead"],
        num_encoder_layers=cfg["num_layers"], num_decoder_layers=cfg["num_layers"],
        dim_feedforward=cfg["dim_feedforward"], dropout=cfg["dropout"],
    )


In [20]:
# 2-1b. 벤치마크 전에 미리 최소 버전 선언

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


class Seq2SeqTransformer(nn.Module):
    # PyTorch layer로 구현

    def __init__(self, src_vocab, tgt_vocab, d_model=256, nhead=8,
                 num_encoder_layers=4, num_decoder_layers=4,
                 dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.src_tok_emb = nn.Embedding(src_vocab, d_model, padding_idx=PAD_ID)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab, d_model, padding_idx=PAD_ID)
        self.pos_enc = PositionalEncoding(d_model, dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers, num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True,
        )
        self.generator = nn.Linear(d_model, tgt_vocab)

        # weight tying: 디코더 임베딩과 출력 projection이 가중치를 공유
        # (파라미터 절약 + 소규모 데이터/짧은 학습에서 수렴이 빨라짐)
        self.generator.weight = self.tgt_tok_emb.weight

        # 임베딩 초기화: nn.Embedding 기본값은 N(0,1)인데, forward에서
        # sqrt(d_model)을 곱하므로 임베딩 출력 std가 ~16이 되어 학습 초기가 불안정해짐.
        # std=d_model^-0.5로 초기화하면 스케일링 후 std≈1이 되도록 맞춰짐.
        nn.init.normal_(self.src_tok_emb.weight, mean=0.0, std=d_model ** -0.5)
        nn.init.normal_(self.tgt_tok_emb.weight, mean=0.0, std=d_model ** -0.5)
        with torch.no_grad():
            self.src_tok_emb.weight[PAD_ID].fill_(0)
            self.tgt_tok_emb.weight[PAD_ID].fill_(0)

    def forward(self, src, tgt, src_pad_mask, tgt_pad_mask, tgt_mask):
        src_emb = self.pos_enc(self.src_tok_emb(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_enc(self.tgt_tok_emb(tgt) * math.sqrt(self.d_model))
        out = self.transformer(
            src_emb, tgt_emb, tgt_mask=tgt_mask,
            src_key_padding_mask=src_pad_mask, tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )
        return self.generator(out)

    def encode(self, src, src_pad_mask):
        src_emb = self.pos_enc(self.src_tok_emb(src) * math.sqrt(self.d_model))
        return self.transformer.encoder(src_emb, src_key_padding_mask=src_pad_mask)

    def decode(self, tgt, memory, tgt_mask, src_pad_mask):
        tgt_emb = self.pos_enc(self.tgt_tok_emb(tgt) * math.sqrt(self.d_model))
        return self.transformer.decoder(tgt_emb, memory, tgt_mask=tgt_mask, memory_key_padding_mask=src_pad_mask)


def causal_mask(sz, device):
    # bool 마스크 사용 (key_padding_mask도 bool이라 dtype을 맞춰줘야 경고가 안 뜸)
    return torch.triu(torch.ones(sz, sz, dtype=torch.bool, device=device), diagonal=1)


In [21]:
# 2-2. 실제 벤치마크 실행

def benchmark_config(name, cfg):
    model = build_model(cfg, BENCH_VOCAB, BENCH_VOCAB).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    src = torch.randint(4, BENCH_VOCAB, (cfg["batch_size"], BENCH_SEQ_LEN), device=device)
    tgt = torch.randint(4, BENCH_VOCAB, (cfg["batch_size"], BENCH_SEQ_LEN), device=device)
    tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
    src_pad_mask = torch.zeros_like(src, dtype=torch.bool)
    tgt_pad_mask = torch.zeros_like(tgt_in, dtype=torch.bool)
    tmask = causal_mask(tgt_in.size(1), device)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    # 워밍업
    for _ in range(2):
        opt.zero_grad()
        logits = model(src, tgt_in, src_pad_mask, tgt_pad_mask, tmask)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        loss.backward()
        opt.step()
    if device.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(BENCH_STEPS):
        opt.zero_grad()
        logits = model(src, tgt_in, src_pad_mask, tgt_pad_mask, tmask)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        loss.backward()
        opt.step()
    if device.type == "cuda":
        torch.cuda.synchronize()
    sec_per_step = (time.time() - t0) / BENCH_STEPS
    n_params = sum(p.numel() for p in model.parameters())
    del model, opt
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return n_params, sec_per_step


DATASET_SIZE_CANDIDATES = [20_000, 50_000, 100_000, 300_000, len(raw)]

rows = []
bench_results = {}
for name, cfg in CANDIDATE_CONFIGS.items():
    n_params, sec_per_step = benchmark_config(name, cfg)
    bench_results[name] = sec_per_step
    row = {
        "config": name,
        "d_model": cfg["d_model"], "layers": cfg["num_layers"], "heads": cfg["nhead"],
        "batch_size": cfg["batch_size"], "params(M)": round(n_params / 1e6, 2),
        "sec/step": round(sec_per_step, 3),
    }
    for n in DATASET_SIZE_CANDIDATES:
        steps_per_epoch = math.ceil(n / cfg["batch_size"])
        est_minutes = steps_per_epoch * sec_per_step / 60
        row[f"~{n:,}쌍 1epoch(분)"] = round(est_minutes, 1)
    rows.append(row)

bench_df = pd.DataFrame(rows).set_index("config")
pd.set_option("display.width", 160)
print(f"device = {device}")
bench_df


device = cuda


,d_model,layers,heads,batch_size,params(M),sec/step,"~20,000쌍 1epoch(분)","~50,000쌍 1epoch(분)","~100,000쌍 1epoch(분)","~300,000쌍 1epoch(분)","~1,690,052쌍 1epoch(분)"
config,,,,,,,,,,,
tiny,128,2,4,128,2.98,0.035,0.1,0.2,0.5,1.4,7.6
small,256,3,4,128,9.63,0.090,0.2,0.6,1.2,3.5,19.9
base,256,4,8,64,11.48,0.062,0.3,0.8,1.6,4.9,27.4
large,512,6,8,64,52.34,0.258,1.3,3.4,6.7,20.1,113.4


### 2-3. 데이터 규모 / 모델 설정 선택

`sec/step`과 `~N쌍 1epoch(분)` 열을 참고


In [ ]:
NUM_TRAIN_EXAMPLES = 1_000_000   # 학습에 사용할 문장쌍 수
SELECTED_CONFIG_NAME = "base"    # "tiny" | "small" | "base" | "large"
VOCAB_SIZE = 8000                # BPE 토크나이저 vocab 크기 (ko/en 각각)
MAX_LEN = 64                     # 토큰 기준 최대 길이 (초과분은 truncate)
VAL_SIZE = 2000
TEST_SIZE = 2000

CFG = CANDIDATE_CONFIGS[SELECTED_CONFIG_NAME]
est_minutes = math.ceil(NUM_TRAIN_EXAMPLES / CFG["batch_size"]) * bench_results[SELECTED_CONFIG_NAME] / 60
print(f"선택: {SELECTED_CONFIG_NAME} {CFG}")
print(f"학습 {NUM_TRAIN_EXAMPLES:,}쌍 기준 예상 1-epoch 학습 시간: 약 {est_minutes:.1f}분 (device={device}, unlikelihood loss 사용 시 +30% 정도)")


선택: base {'d_model': 256, 'nhead': 8, 'num_layers': 4, 'dim_feedforward': 1024, 'dropout': 0.1, 'batch_size': 64, 'lr': 0.0003}
학습 1,000,000쌍 기준 예상 1-epoch 학습 시간: 약 16.2분 (device=cuda, unlikelihood loss 사용 시 +30% 정도)


## 3. Train/Val/Test 분할

In [23]:
total_needed = NUM_TRAIN_EXAMPLES + VAL_SIZE + TEST_SIZE
assert total_needed <= len(raw), "요청한 데이터 수가 전체 데이터셋보다 많습니다."

subset = raw.select(range(total_needed))
train_split = subset.select(range(0, NUM_TRAIN_EXAMPLES))
val_split = subset.select(range(NUM_TRAIN_EXAMPLES, NUM_TRAIN_EXAMPLES + VAL_SIZE))
test_split = subset.select(range(NUM_TRAIN_EXAMPLES + VAL_SIZE, total_needed))

print("train:", len(train_split), "val:", len(val_split), "test:", len(test_split))


train: 1000000 val: 2000 test: 2000


## 4. 토크나이저 학습 (Byte-level BPE)

사전학습된 토크나이저를 쓰지 않고, 학습 데이터로 ko/en 각각의 BPE 토크나이저를 직접 학습합니다.


In [24]:
from tokenizers import ByteLevelBPETokenizer
from tokenizers.processors import TemplateProcessing

SPECIAL_TOKENS = ["<pad>", "<sos>", "<eos>", "<unk>"]
PAD_ID, SOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

TOKENIZER_TRAIN_CAP = min(len(train_split), 200_000)

def train_bpe(sentences, vocab_size):
    tok = ByteLevelBPETokenizer()
    tok.train_from_iterator(sentences, vocab_size=vocab_size, min_frequency=2, special_tokens=SPECIAL_TOKENS)
    tok.post_processor = TemplateProcessing(
        single="<sos> $A <eos>",
        special_tokens=[("<sos>", tok.token_to_id("<sos>")), ("<eos>", tok.token_to_id("<eos>"))],
    )
    tok.enable_truncation(max_length=MAX_LEN)
    return tok

ko_tokenizer = train_bpe(train_split["ko"][:TOKENIZER_TRAIN_CAP], VOCAB_SIZE)
en_tokenizer = train_bpe(train_split["en"][:TOKENIZER_TRAIN_CAP], VOCAB_SIZE)

print("ko vocab:", ko_tokenizer.get_vocab_size(), "en vocab:", en_tokenizer.get_vocab_size())
sample_enc = ko_tokenizer.encode(train_split["ko"][0])

# 길이 분포 확인 -> MAX_LEN 적절성 체크
lens = [len(ko_tokenizer.encode(s).ids) for s in train_split["ko"][:2000]]
print(f"ko 토큰 길이: p50={np.percentile(lens,50):.0f} p95={np.percentile(lens,95):.0f} p99={np.percentile(lens,99):.0f} (MAX_LEN={MAX_LEN})")


ko vocab: 8000 en vocab: 8000
ko 토큰 길이: p50=13 p95=24 p99=31 (MAX_LEN=64)


## 5. Dataset / DataLoader

In [25]:
class TranslationDataset(Dataset):
    def __init__(self, hf_split, src_tok, tgt_tok):
        self.src_ids = [src_tok.encode(s).ids for s in hf_split["ko"]]
        self.tgt_ids = [tgt_tok.encode(s).ids for s in hf_split["en"]]

    def __len__(self):
        return len(self.src_ids)

    def __getitem__(self, idx):
        return torch.tensor(self.src_ids[idx]), torch.tensor(self.tgt_ids[idx])


def collate_fn(batch):
    srcs, tgts = zip(*batch)
    src_max = max(len(s) for s in srcs)
    tgt_max = max(len(t) for t in tgts)
    src_pad = torch.full((len(srcs), src_max), PAD_ID, dtype=torch.long)
    tgt_pad = torch.full((len(tgts), tgt_max), PAD_ID, dtype=torch.long)
    for i, s in enumerate(srcs):
        src_pad[i, : len(s)] = s
    for i, t in enumerate(tgts):
        tgt_pad[i, : len(t)] = t
    return src_pad, tgt_pad


train_ds = TranslationDataset(train_split, ko_tokenizer, en_tokenizer)
val_ds = TranslationDataset(val_split, ko_tokenizer, en_tokenizer)
test_ds = TranslationDataset(test_split, ko_tokenizer, en_tokenizer)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate_fn)

print("train batches:", len(train_loader), "val batches:", len(val_loader), "test batches:", len(test_loader))


train batches: 15625 val batches: 32 test batches: 32


## 6. Transformer 모델 (최종)

In [26]:
def make_masks(src, tgt_in):
    src_pad_mask = (src == PAD_ID)
    tgt_pad_mask = (tgt_in == PAD_ID)
    tgt_mask = causal_mask(tgt_in.size(1), src.device)
    return src_pad_mask, tgt_pad_mask, tgt_mask


model = build_model(CFG, ko_tokenizer.get_vocab_size(), en_tokenizer.get_vocab_size()).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"model params: {n_params/1e6:.2f}M")
print(model)
# `Embedding`/`Transformer`/`Linear`/`Dropout` 레이어로 구성

model params: 11.48M
Seq2SeqTransformer(
  (src_tok_emb): Embedding(8000, 256, padding_idx=0)
  (tgt_tok_emb): Embedding(8000, 256, padding_idx=0)
  (pos_enc): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=1024, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=1024, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (n

## 7. 학습 (1 epoch)

In [27]:
import torch.nn.functional as F

EPOCHS = 1            # 과제 규칙: 1 epoch 고정
WARMUP_RATIO = 0.06
LOG_EVERY = 50
UL_ALPHA = 0.5        # unlikelihood loss 가중치 (0이면 기존 MLE 학습과 동일)

total_steps = len(train_loader) * EPOCHS
warmup_steps = max(50, int(total_steps * WARMUP_RATIO))

optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"], betas=(0.9, 0.98), eps=1e-9)


# linear warmup + linear decay 스케줄 (순수 PyTorch — HF transformers 의존성 제거)
def linear_warmup_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, linear_warmup_lambda)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

# AMP(mixed precision): GPU에서 ~1.5-2배 빠르고 메모리 절약 (CPU에선 자동으로 비활성)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(enabled=use_amp)


def unlikelihood_loss(logits, tgt_out):
    """Token-level unlikelihood training (비선호 학습, Welleck et al. 2019).

    타깃 문장에서 '현재 위치보다 앞에 이미 등장한 토큰'을 반복 후보(negative candidate)로
    보고, 그 토큰들의 확률을 낮추는 -log(1 - p) 페널티를 추가한다.
    MLE(CE)만으로 학습하면 자주 나온 토큰을 계속 뱉는 반복 루프가 생기기 쉬운데,
    이 loss가 반복 생성 확률 자체를 학습 단계에서 억제한다.
    """
    B, T, V = logits.shape
    logprobs = F.log_softmax(logits.float(), dim=-1)

    # prev[b, t, i] = tgt_out[b, i]  (i < t 위치만 유효, i >= t는 pad로 밀어 무효화)
    prev = tgt_out.unsqueeze(1).expand(B, T, T)
    future = torch.triu(torch.ones(T, T, dtype=torch.bool, device=logits.device))
    prev = prev.masked_fill(future.unsqueeze(0), PAD_ID)

    # cand[b, t, v] = True  <=>  v가 t 이전에 등장한 토큰 (pad/현재 정답 제외)
    cand = torch.zeros(B, T, V, dtype=torch.bool, device=logits.device)
    cand.scatter_(2, prev, True)
    cand[:, :, PAD_ID] = False                     # pad는 후보 아님
    cand.scatter_(2, tgt_out.unsqueeze(2), False)  # 현재 정답 토큰은 페널티 제외
    cand &= (tgt_out != PAD_ID).unsqueeze(2)       # pad 위치는 loss 계산 안 함

    one_minus_p = (1.0 - logprobs.exp()).clamp_min(1e-6)
    n_tok = (tgt_out != PAD_ID).sum().clamp_min(1)
    return -(one_minus_p.log() * cand.float()).sum() / n_tok


@torch.no_grad()
def evaluate_loss(loader):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
        src_pad_mask, tgt_pad_mask, tgt_mask = make_masks(src, tgt_in)
        logits = model(src, tgt_in, src_pad_mask, tgt_pad_mask, tgt_mask)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        n_tok = (tgt_out != PAD_ID).sum().item()
        total_loss += loss.item() * n_tok
        total_tokens += n_tok
    model.train()
    return total_loss / total_tokens


model.train()
t0 = time.time()
step = 0
running_ce, running_ul = 0.0, 0.0
for epoch in range(EPOCHS):
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
        src_pad_mask, tgt_pad_mask, tgt_mask = make_masks(src, tgt_in)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            logits = model(src, tgt_in, src_pad_mask, tgt_pad_mask, tgt_mask)
            ce_loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            ul_loss = unlikelihood_loss(logits, tgt_out) if UL_ALPHA > 0 else logits.new_zeros(())
            loss = ce_loss + UL_ALPHA * ul_loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_ce += ce_loss.item()
        running_ul += ul_loss.item()
        step += 1
        if step % LOG_EVERY == 0:
            elapsed = time.time() - t0
            eta = elapsed / step * (total_steps - step)
            print(f"step {step}/{total_steps}  ce {running_ce/LOG_EVERY:.4f}  ul {running_ul/LOG_EVERY:.4f}  "
                  f"lr {scheduler.get_last_lr()[0]:.2e}  elapsed {elapsed/60:.1f}m  eta {eta/60:.1f}m")
            running_ce, running_ul = 0.0, 0.0

val_loss = evaluate_loss(val_loader)
print(f"\n1 epoch 완료. val loss={val_loss:.4f}  val PPL={math.exp(val_loss):.2f}  총 소요 {(time.time()-t0)/60:.1f}분")


step 50/15625  ce 9.0230  ul 0.0028  lr 1.60e-05  elapsed 0.0m  eta 13.6m
step 100/15625  ce 8.0275  ul 0.0093  lr 3.20e-05  elapsed 0.1m  eta 14.2m
step 150/15625  ce 7.4723  ul 0.0149  lr 4.80e-05  elapsed 0.1m  eta 13.7m
step 200/15625  ce 7.0254  ul 0.0201  lr 6.40e-05  elapsed 0.2m  eta 13.4m
step 250/15625  ce 6.6918  ul 0.0279  lr 8.00e-05  elapsed 0.2m  eta 13.2m
step 300/15625  ce 6.4935  ul 0.0328  lr 9.61e-05  elapsed 0.3m  eta 13.2m
step 350/15625  ce 6.3561  ul 0.0334  lr 1.12e-04  elapsed 0.3m  eta 13.3m
step 400/15625  ce 6.2577  ul 0.0318  lr 1.28e-04  elapsed 0.3m  eta 13.2m
step 450/15625  ce 6.1724  ul 0.0299  lr 1.44e-04  elapsed 0.4m  eta 13.1m
step 500/15625  ce 6.0728  ul 0.0275  lr 1.60e-04  elapsed 0.4m  eta 13.0m
step 550/15625  ce 5.9996  ul 0.0266  lr 1.76e-04  elapsed 0.5m  eta 13.1m
step 600/15625  ce 5.9165  ul 0.0250  lr 1.92e-04  elapsed 0.5m  eta 13.0m
step 650/15625  ce 5.8442  ul 0.0251  lr 2.08e-04  elapsed 0.6m  eta 12.9m
step 700/15625  ce 5.7978 

## 8. 평가

In [28]:
test_loss = evaluate_loss(test_loader)
print(f"test loss={test_loss:.4f}  test PPL={math.exp(test_loss):.2f}")


test loss=3.5166  test PPL=33.67


In [29]:
@torch.no_grad()
def batch_translate(sentences, batch_size=32, max_len=MAX_LEN, num_beams=4,
                    length_penalty=0.6, repetition_penalty=1.2, no_repeat_ngram_size=3):
    """배치 beam search 디코딩 (num_beams=1이면 greedy와 동일).

    추론 하이퍼파라미터 (반복/환각 억제용):
    - num_beams: beam 폭. 클수록 더 많은 후보 탐색 (greedy의 근시안적 선택 완화)
    - length_penalty: 길이 보정 (GNMT식). 낮을수록 짧은 번역 선호 완화
    - repetition_penalty: 이미 생성한 토큰의 logit 약화 (CTRL식) -> 반복 억제
    - no_repeat_ngram_size: 같은 n-gram이 두 번 등장하지 못하게 차단 -> 반복 루프 차단
    """
    model.eval()
    results = []
    k = num_beams
    for i in range(0, len(sentences), batch_size):
        chunk = sentences[i : i + batch_size]
        encs = [ko_tokenizer.encode(s).ids for s in chunk]
        B = len(encs)
        src = torch.full((B, max(len(e) for e in encs)), PAD_ID, dtype=torch.long, device=device)
        for j, e in enumerate(encs):
            src[j, : len(e)] = torch.tensor(e, device=device)
        src_pad_mask = (src == PAD_ID)
        memory = model.encode(src, src_pad_mask)

        # 빔 차원으로 확장: (B*k, ...)
        mem = memory.repeat_interleave(k, dim=0)
        mem_pad = src_pad_mask.repeat_interleave(k, dim=0)

        ys = torch.full((B * k, 1), SOS_ID, dtype=torch.long, device=device)
        beam_scores = torch.full((B, k), float("-inf"), device=device)
        beam_scores[:, 0] = 0.0  # 첫 step엔 모든 빔이 동일하므로 하나만 활성화
        beam_scores = beam_scores.view(-1)
        finished = torch.zeros(B * k, dtype=torch.bool, device=device)

        for _ in range(max_len - 1):
            tmask = causal_mask(ys.size(1), device)
            out = model.decode(ys, mem, tmask, mem_pad)
            logits = model.generator(out[:, -1]).float()  # (B*k, V)
            V = logits.size(-1)

            # repetition penalty: 이미 생성된 토큰의 logit 약화
            if repetition_penalty != 1.0:
                gen_scores = logits.gather(1, ys)
                gen_scores = torch.where(gen_scores > 0, gen_scores / repetition_penalty,
                                         gen_scores * repetition_penalty)
                logits.scatter_(1, ys, gen_scores)

            # no-repeat n-gram: 직전 (n-1)개 토큰과 같은 접두를 가진 n-gram의 마지막 토큰 금지
            if no_repeat_ngram_size > 0 and ys.size(1) >= no_repeat_ngram_size:
                n = no_repeat_ngram_size
                for b in range(B * k):
                    if finished[b]:
                        continue
                    seq = ys[b].tolist()
                    prefix = tuple(seq[len(seq) - n + 1:])
                    banned = [seq[j + n - 1] for j in range(len(seq) - n + 1)
                              if tuple(seq[j : j + n - 1]) == prefix]
                    if banned:
                        logits[b, banned] = float("-inf")

            logprobs = torch.log_softmax(logits, dim=-1)
            # 끝난 빔은 pad만 이어붙이며 누적 score 유지
            logprobs[finished] = float("-inf")
            logprobs[finished, PAD_ID] = 0.0

            next_scores = (beam_scores.unsqueeze(1) + logprobs).view(B, k * V)
            top_scores, top_idx = next_scores.topk(k, dim=1)          # (B, k)
            beam_idx = torch.div(top_idx, V, rounding_mode="floor")   # 어느 빔에서 왔는지
            tok_idx = top_idx % V

            src_beam = (torch.arange(B, device=device).unsqueeze(1) * k + beam_idx).view(-1)
            ys = torch.cat([ys[src_beam], tok_idx.view(-1, 1)], dim=1)
            beam_scores = top_scores.view(-1)
            finished = finished[src_beam] | (tok_idx.view(-1) == EOS_ID)
            if finished.all():
                break

        # length penalty로 보정한 score가 가장 높은 빔 선택
        ys_v = ys.view(B, k, -1)
        lengths = (ys_v != PAD_ID).sum(-1).float()
        lp = ((5.0 + lengths) / 6.0) ** length_penalty
        best = (beam_scores.view(B, k) / lp).argmax(dim=1)
        for b in range(B):
            results.append(en_tokenizer.decode(ys_v[b, best[b]].tolist(), skip_special_tokens=True))
    return results


def translate(sentence, **kwargs):
    # 단일 문장 번역 (데모용)
    return batch_translate([sentence], **kwargs)[0]


# ---- 추론 하이퍼파라미터 탐색: 디코딩 설정별 BLEU 비교 ----
EVAL_SAMPLES = min(len(test_split), 1000)
srcs = test_split["ko"][:EVAL_SAMPLES]
refs = test_split["en"][:EVAL_SAMPLES]

import sacrebleu

DECODE_SETTINGS = {
    "greedy (기존)":    dict(num_beams=1, repetition_penalty=1.0, no_repeat_ngram_size=0),
    "greedy+반복억제":  dict(num_beams=1, repetition_penalty=1.2, no_repeat_ngram_size=3),
    "beam4+반복억제":   dict(num_beams=4, length_penalty=0.6, repetition_penalty=1.2, no_repeat_ngram_size=3),
}

all_hyps = {}
for name, kw in DECODE_SETTINGS.items():
    t0 = time.time()
    all_hyps[name] = batch_translate(srcs, **kw)
    score = sacrebleu.corpus_bleu(all_hyps[name], [refs]).score
    print(f"{name:18s} BLEU {score:5.2f}  ({time.time()-t0:.0f}초, {EVAL_SAMPLES}문장)")

hyps = all_hyps["beam4+반복억제"]  # 아래 정성 예시 출력용


greedy (기존)        BLEU 14.47  (6초, 1000문장)
greedy+반복억제        BLEU 14.27  (5초, 1000문장)
beam4+반복억제         BLEU 14.98  (11초, 1000문장)


In [30]:
# 
examples = pd.DataFrame({"src(ko)": srcs[:10], "ref(en)": refs[:10], "pred(en)": hyps[:10]})
pd.set_option("display.max_colwidth", 80)
examples


,src(ko),ref(en),pred(en)
0,"오, 조지아에서 생산된 묘목은 제 농장에서 살아남지 못할 거예요.",Seedlings produced in Georgia won't survive on my farm.,"Oh, I will not be able to live in my farm at the end."
1,저는 일반적으로 댓글에 응답하고 캠페인을 개념화하고 다양한 콘텐츠를 만드는 일을 합니다.,"I am usually the one who responds to comments, conceptualizes campaigns, and...",I usually respond to my house and make a variety of content on the campaign.
2,검은색 프린트 티셔츠는 라운드 넥 디자인입니다.,The black printed t-shirt is a round neck.,It is a black-shirt design.
3,이러한 기술은 사용자에게 믿을 수 없을 정도로 안전한 주행 및 여행 경험을 제공합니다.,Such technology creates an incredibly safe driving and traveling experiences...,These technology provides a safe experience and travel experience that can't...
4,아침에 버스 타려고 막 뛰다가 다리를 삐긋했어요.,I was running to catch a bus in the morning and sprained my leg.,"I tried to take a bus in the morning, and I bought it."
5,"알겠어, 확인만 하고 싶어.","Okay, I just want to confirm it.","OK, I just want to check out."
6,모던에서 클래식까지 트렌디한 디자인을 제안합니다.,We offer trendy designs from modern to classic.,We offer a modern design that is made from the day.
7,"지금 필요하신 두께를 알려 주셔야 제작 가능 여부, 가격 알려드릴 수 있어요.","Please tell us the thickness now, so that we can inform you of whether it's ...","If you need to inform me of the two options, I can tell you the price."
8,올바른 정책은 올바른 예측이 기본이 되어야 합니다.,Right policies must be based on the right predictions.,The right policy has to be the correct process.
9,"안녕하세요, 체크인 하고 싶어요.","HI, I would like to check in.","Hello, I would like to check out."
